# Prerequisites

## 1. Reading a CSV file

In [0]:
df = spark.read.csv(
    path = "/Volumes/merit_catalog/quickstart_schema/sandbox/dataset/user_dataset/users_001.csv",
    header = True,
    inferSchema = True
)

## 2. Writing it to Delta format

In [0]:
df.write.format("delta").save(
    path="/Volumes/merit_catalog/quickstart_schema/sandbox/output/output_delta",
    mode="overwrite",
)

## 3. Loading the Delta file

In [0]:
df_delta = spark.read.load(
    "/Volumes/merit_catalog/quickstart_schema/sandbox/output/output_delta"
)
df_delta.limit(6).display()

## 4. Trying to update data
Using SQL API, we try to update the table. For that, we have to create a view for the table

In [0]:
# Creating a view
df_delta.createOrReplaceTempView("user_vw")

In [0]:
%sql
UPDATE user_vw
SET country = "Bharat"
WHERE country = "India"

In [0]:
%sql
UPDATE user_vw
SET country = "India"
WHERE country = "Bharat"

In [0]:
df_delta.display()

The below attempt to overwrite the file with new schema'd file, it says mismatch schema.

In [0]:
spark.read.csv(
    path="/Volumes/merit_catalog/quickstart_schema/sandbox/dataset/user_dataset/users_006_new_column_education.csv",
    header="True",
    inferSchema=True,
).write.format("delta").save("/Volumes/merit_catalog/quickstart_schema/sandbox/output/output_delta", mode = "overwrite")

## Writing new data with same schema, only of country India

In [0]:
from pyspark.sql.functions import col

df.filter(col("country") == "India").write.format("delta").save(
    path="/Volumes/merit_catalog/quickstart_schema/sandbox/output/output_delta",
    mode="overwrite",
)

## Reading Delta data

In [0]:
spark.read.load(
    "/Volumes/merit_catalog/quickstart_schema/sandbox/output/output_delta"
).limit(5).display()

## Loading the log file

### Approach 1

In [0]:
spark.read.text("/Volumes/merit_catalog/quickstart_schema/sandbox/output/output_delta/_delta_log/00000000000000000000.json").display()

### Approach 2

In [0]:
from delta.tables import DeltaTable
delta_table = DeltaTable.forPath(spark, "/Volumes/merit_catalog/quickstart_schema/sandbox/output/output_delta")

delta_table.history().display()